# SwinV2 — TTA on Existing Checkpoint
Run all cells top to bottom. No training required.

In [1]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import cohen_kappa_score, classification_report, f1_score
from PIL import Image
from tqdm.auto import tqdm
print("Imports done.")

/scratch/AML_G14_CW/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports done.


In [2]:
# ── Paths — same as your existing notebooks ───────────────────────────────────
NOTEBOOK_DIR    = Path().resolve()
ROOT            = NOTEBOOK_DIR.parent
DATA_ROOT       = ROOT / 'data' / 'diabetic-retinopathy-detection'
VAL_DIR         = DATA_ROOT / 'val_proc'
TEST_LABELS_DIR = DATA_ROOT / 'test_labels_proc'
VAL_CSV         = DATA_ROOT / 'val_split.csv'
TEST_CSV        = DATA_ROOT / 'test_split.csv'

# ── Your best checkpoint — change this if needed ──────────────────────────────
BASELINE_CKPT = ROOT / 'outputs' / 'swinv2_small' / 'checkpoints' / 'swinv2_small_best.pt'

MODEL_NAME  = 'swinv2_small_window8_256'
WEIGHTS_PATH = ROOT / 'weights' / 'swinv2_small.safetensors'
SEED        = 42
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
IMAGE_SIZE  = 256
SWIN_MEAN   = (0.485, 0.456, 0.406)
SWIN_STD    = (0.229, 0.224, 0.225)
BATCH_SIZE  = 16
NUM_WORKERS = 4
DROPOUT     = 0.3

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Checkpoint: {BASELINE_CKPT}')
print(f'Exists: {BASELINE_CKPT.exists()}')

Device: cuda
Checkpoint: /scratch/AML_G14_CW/outputs/swinv2_small/checkpoints/swinv2_small_best.pt
Exists: True


In [3]:
# ── Dataset + loaders ─────────────────────────────────────────────────────────
class DRDataset(Dataset):
    def __init__(self, df, data_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = Image.open(self.data_dir / f"{row['image']}.jpeg").convert('RGB')
        label = int(row['level'])
        if self.transform: image = self.transform(image)
        return image, label

val_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.05)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(SWIN_MEAN, SWIN_STD),
])

val_df  = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

val_loader = DataLoader(
    DRDataset(val_df, VAL_DIR, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(
    DRDataset(test_df, TEST_LABELS_DIR, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)

print(f'Val  : {len(val_df):,} images')
print(f'Test : {len(test_df):,} images')

Val  : 5,269 images
Test : 5,269 images


In [4]:
# ── Build model + load checkpoint ─────────────────────────────────────────────
backbone = timm.create_model(MODEL_NAME, pretrained=False,
                              num_classes=0, global_pool='avg')
feat_dim = backbone.num_features
backbone.custom_head = nn.Sequential(
    nn.LayerNorm(feat_dim), nn.Dropout(DROPOUT),
    nn.Linear(feat_dim, NUM_CLASSES))
_base = backbone.forward
backbone.forward = lambda x: backbone.custom_head(_base(x))
model = backbone.to(device)

ckpt = torch.load(BASELINE_CKPT, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
model.eval()
print(f'Loaded checkpoint: epoch={ckpt["epoch"]}  val_κ={ckpt["val_kappa"]:.4f}')

Loaded checkpoint: epoch=27  val_κ=0.7034


In [5]:
# ── Standard eval (no TTA) ────────────────────────────────────────────────────
std_preds, std_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Standard eval'):
        out = model(images.to(device))
        std_preds.extend(out.argmax(1).cpu().tolist())
        std_labels.extend(labels.tolist())

std_kappa = cohen_kappa_score(std_labels, std_preds, weights='quadratic')
print(f'Standard QWK: {std_kappa:.4f}')

Standard eval: 100%|██████████| 330/330 [00:42<00:00,  7.74it/s]

Standard QWK: 0.7136


In [6]:
# ── TTA — 5 views, no training needed ─────────────────────────────────────────
tta_transforms = [
    lambda x: x,
    lambda x: torch.flip(x, dims=[3]),
    lambda x: torch.flip(x, dims=[2]),
    lambda x: torch.rot90(x, k=1, dims=[2, 3]),
    lambda x: torch.rot90(x, k=3, dims=[2, 3]),
]

model.eval()
tta_preds, tta_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='TTA (5 views)'):
        images = images.to(device)
        probs  = torch.stack([
            F.softmax(model(tfm(images)), dim=1)
            for tfm in tta_transforms
        ]).mean(0)
        tta_preds.extend(probs.argmax(1).cpu().tolist())
        tta_labels.extend(labels.tolist())

tta_kappa = cohen_kappa_score(tta_labels, tta_preds, weights='quadratic')

print(f'\n{"="*50}')
print(f'  Standard QWK : {std_kappa:.4f}')
print(f'  TTA QWK      : {tta_kappa:.4f}  (Δ = {tta_kappa-std_kappa:+.4f})')
print(f'{"="*50}')
print()
print('Per-class F1 (TTA):')
f1s = f1_score(tta_labels, tta_preds, average=None, labels=list(range(5)))
for name, f1 in zip(CLASS_NAMES, f1s):
    bar = '█' * int(f1 * 20)
    print(f'  {name:<18}: {f1:.4f}  {bar}')
print()
print(classification_report(tta_labels, tta_preds,
                             target_names=CLASS_NAMES, digits=3))

TTA (5 views): 100%|██████████| 330/330 [02:36<00:00,  2.11it/s]



  Standard QWK : 0.7136
  TTA QWK      : 0.7247  (Δ = +0.0111)

Per-class F1 (TTA):
  No DR             : 0.9083  ██████████████████
  Mild              : 0.0160  
  Moderate          : 0.5845  ███████████
  Severe            : 0.4245  ████████
  Proliferative     : 0.6337  ████████████

               precision    recall  f1-score   support

        No DR      0.867     0.954     0.908      3872
         Mild      0.333     0.008     0.016       366
     Moderate      0.575     0.594     0.585       794
       Severe      0.556     0.344     0.425       131
Proliferative      0.667     0.604     0.634       106

     accuracy                          0.812      5269
    macro avg      0.599     0.501     0.513      5269
 weighted avg      0.774     0.812     0.780      5269

